<a href="https://colab.research.google.com/github/shelxty/technica_reading_line/blob/main/technica_readingline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install opencv-python mediapipe numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import cv2
import numpy as np
import mediapipe as mp

In [3]:
# configuration stuffs
CAM_INDEX = 0
FRAME_W, FRAME_H = 1280, 720 # basically 1280x720px
ZOOM = 3.0

STRIP_W, STRIP_H = 900, 220 # this is the strip window content size in pixels
SWATCH_H = 44 # color picker bar height
PROFILE_HALF_WIDTH = 400 # this is the amount of pixels left/right of the fingertip that's used to find the line
SWITCH_FRAMES = 4 # number of frames a new line must win before switching
MAX_PICK_DIST = 90 # ignore anything farteher than this from the fingertip
SMOOTH_ALPHA = 0.4 # fingertip smoothing
# ^ for smooth alpha, higher = snappier fingertip smoothing

SCHEMES = [  # color schemes yay
    ("Black / White", (0, 0, 0), (255, 255, 255)),
    ("White / Black", (255, 255, 255), (0, 0, 0)),
    ("Yellow / Black", (0, 255, 255), (0, 0, 0)),
    ("Black / Yellow", (0, 0, 0), (150, 255, 255)),
    ("White / Blue", (255, 255, 255), (120, 40, 0)),
]

scheme_idx = 0 # this is the currently selected scheme


In [9]:
!pip install mediapipe==0.10.14
import mediapipe as mp

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35, 1.0.0, 1.0.1, 1.1.0, 1.1.1)
ERROR: No matching distribution found for mediapipe==0.10.14


In [10]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Download the model file (Tasks API needs it explicitly)
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.IMAGE,   # use VIDEO for frame sequences
    num_hands=1,                             # was max_num_hands
    min_hand_detection_confidence=0.6,       # was min_detection_confidence
    min_tracking_confidence=0.5,
)
landmarker = vision.HandLandmarker.create_from_options(options)

In [4]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands = 1,
    min_detection_confidence = 0.6, # self-explanatory
    min_tracking_confidence = 0.5,
)

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [5]:
import cv2
import numpy as np
import mediapipe as mp

# Initialize MediaPipe Hands and drawing utilities
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Start webcam
cap = cv2.VideoCapture(0)

# Initialize Hands model
hands = mp_hands.Hands(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip for selfie view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect hands
    results = hands.process(rgb_frame)

    # Draw landmarks
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

    cv2.imshow("Hand Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


AttributeError: module 'mediapipe' has no attribute 'solutions'

In [6]:
pip show mediapipe

Name: mediapipe
Version: 1.1.1
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web. See the privacy notice at https://goo.gle/mediapipe-privacy.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: absl-py, certifi, flatbuffers, matplotlib, numpy, opencv-contrib-python, sounddevice
Required-by: 


In [13]:
import time
import urllib.request
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Download the model once (works locally and in Colab)
MODEL_PATH = "hand_landmarker.task"
URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
try:
    open(MODEL_PATH, "rb").close()
except FileNotFoundError:
    urllib.request.urlretrieve(URL, MODEL_PATH)

# Landmark index pairs forming the hand skeleton
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # index
    (5, 9), (9, 10), (10, 11), (11, 12),    # middle
    (9, 13), (13, 14), (14, 15), (15, 16),  # ring
    (13, 17), (17, 18), (18, 19), (19, 20), # pinky
    (0, 17),                                # palm edge
]

def draw_landmarks(frame, hand_landmarks):
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 255, 0), 2)
    for p in pts:
        cv2.circle(frame, p, 4, (0, 0, 255), -1)

# Create the landmarker (VIDEO mode for webcam frames)
options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(0)
start = time.time()

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Wrap the frame and give it a strictly increasing timestamp (ms)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int((time.time() - start) * 1000)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                draw_landmarks(frame, hand_landmarks)

        cv2.imshow("Hand Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

In [15]:
!pip install gradio
import gradio as gr

def process(frame):
    # run your landmarker here, draw landmarks on frame, return it
    return frame

gr.Interface(process, gr.Image(sources=["webcam"], streaming=True), "image").launch(debug = True)

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:741: UserWarning: Streaming components are only supported in live interfaces.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://90341468ae7228bb6a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
Error while flagging: field larger than field limit (131072)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fa787753cf6edf18e1.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://90341468ae7228bb6a.gradio.live


In [ ]:
!pip install -q gradio mediapipe opencv-python

import urllib.request, os
import cv2
import numpy as np
import gradio as gr
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Download the model once
MODEL_PATH = "hand_landmarker.task"
URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(URL, MODEL_PATH)

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
]

# Create the landmarker once. IMAGE mode treats each frame independently,
# which suits Gradio, since frames arrive as separate requests.
options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = vision.HandLandmarker.create_from_options(options)

def process(frame):
    if frame is None:
        return None

    frame = np.ascontiguousarray(frame)        # Gradio gives an RGB numpy array
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = landmarker.detect(mp_image)

    out = frame.copy()
    h, w = out.shape[:2]
    for hand in result.hand_landmarks:
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand]
        for a, b in HAND_CONNECTIONS:
            cv2.line(out, pts[a], pts[b], (0, 255, 0), 2)    # green (RGB)
        for p in pts:
            cv2.circle(out, p, 4, (255, 0, 0), -1)           # red (RGB)
    return out

with gr.Blocks() as demo:
    gr.Markdown("## Live hand tracking")
    with gr.Row():
        cam = gr.Image(sources=["webcam"], streaming=True, label="Webcam")
        out = gr.Image(label="Hand landmarks")
    cam.stream(process, inputs=cam, outputs=out, stream_every=0.1, time_limit=60)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://63195ea18ea09508b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
